# Phase 1 — Data + Ingestion

Builds the knowledge base the rest of the pipeline retrieves from: fetch public data, chunk it, embed it, store it.

```mermaid
flowchart LR
    WIKI[Wikipedia] --> RAW1[(raw)]
    WB[World Bank] --> RAW2[(raw)]
    FRED[FRED] --> RAW3[(raw)]
    SEC[SEC EDGAR] --> RAW4[(raw)]
    RAW1 & RAW2 & RAW3 & RAW4 --> CHUNK[chunk + embed] --> PG[(pgvector store)]
```

Below: each source, then chunking, embedding, and storage — real code against the real project.

In [1]:
import sys
sys.path.insert(0, "..")

## The 4 data sources

| Source | What we pull | Why |
|---|---|---|
| **Wikipedia** | 29 articles on financial/economic terms (GDP, Inflation, P/E ratio...) | qualitative context — what a term means |
| **World Bank** | GDP, inflation, unemployment × 8 countries, yearly | quantitative macro facts, multi-country |
| **FRED** | 5 US macro series, monthly/daily since 2015 | denser US-only history, complements World Bank |
| **SEC EDGAR** | Revenue / Net income / Total assets × 8 tickers, from 10-K filings | quantitative corporate facts |

Wikipedia gives prose to explain concepts; the other three give structured, checkable numbers to verify a claim against.

### Wikipedia

Skips the `wikipedia` PyPI package — it sends no `User-Agent`, so Wikipedia returns `403`. Calls the MediaWiki API directly instead.

In [2]:
from ingest.fetch_wikipedia import fetch_wikipedia_article

article = fetch_wikipedia_article("Inflation")
for k, v in article.items():
    print(f"{k}: {v[:300]}")

title: Inflation
url: https://en.wikipedia.org/wiki/Inflation
content: In economics, inflation is an increase in the average price of goods and services in terms of money. This increase is measured using a price index, typically a consumer price index (CPI). When the general price level rises, each unit of currency buys fewer goods and services; consequently, inflation


### World Bank

Structured data: one row per (country, indicator, year), via `fetch_worldbank_series(country, code, label)` — one HTTP call shaped into storage-ready rows.

In [3]:
from ingest.fetch_worldbank import fetch_worldbank_series

rows = fetch_worldbank_series("US", "NY.GDP.MKTP.CD", "GDP (current US$)")
rows[:3]


[{'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2025,
  'value': 30769700000000.0},
 {'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2024,
  'value': 29298013000000.0},
 {'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2023,
  'value': 27811517000000.0}]

### FRED

Same shape as World Bank, via `fetch_fred_series(series_id, label)` in `ingest/fetch_fred.py`. Needs a free `FRED_API_KEY` (see README).

In [4]:
from ingest.fetch_fred import fetch_fred_series

rows = fetch_fred_series("GDP", "Gross Domestic Product")
rows[-3:]

[{'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2025-07-01',
  'value': 31098.027},
 {'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2025-10-01',
  'value': 31422.526},
 {'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2026-01-01',
  'value': 31865.721}]

### SEC EDGAR

Messiest source, two gotchas:

- Revenue's XBRL tag changed with ASC 606 (`Revenues` → `RevenueFromContractWithCustomerExcludingAssessedTax`); using only one tag lost history (Apple: 3 years, not 10). `annual_values()` in `ingest/fetch_secedgar.py` merges both, deduped by period-end date.
- Requires a real `User-Agent` with contact info, like Wikipedia — anonymous requests get rejected.

In [7]:
from ingest.fetch_secedgar import USER_AGENT, fetch_company_facts
from ingest.http import get_json

ticker_map = get_json(
    "https://www.sec.gov/files/company_tickers.json",
    headers={"User-Agent": USER_AGENT},
    timeout=20,
)
rows = fetch_company_facts("AAPL", ticker_map)
rows[:3]

[{'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2016-09-24',
  'value_usd': 215639000000,
  'form': '10-K',
  'filed': '2018-11-05'},
 {'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2017-09-30',
  'value_usd': 229234000000,
  'form': '10-K',
  'filed': '2019-10-31'},
 {'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2018-09-29',
  'value_usd': 265595000000,
  'form': '10-K',
  'filed': '2020-10-30'}]

## Ingestion: dlt

Each source is a [dlt](https://dlthub.com/) resource in `ingest/fetch_*.py`, loaded into its own raw Postgres schema with `write_disposition="merge"` (re-running a fetch doesn't duplicate rows).

```python
@dlt.resource(
    name="worldbank_indicators",
    write_disposition="merge",
    primary_key=["country_code", "indicator_code", "year"]
    )
def worldbank_indicators():
    for country in WB_COUNTRIES:
        for code, label in WB_INDICATORS.items():
            yield from fetch_worldbank_series(country, code, label)
```

**Gotcha:** dlt silently hides type mismatches — e.g. GDP as `int`, inflation as `float` left 1156 of 1226 rows `NULL` in a shadow column. Fix: cast to `float(...)` before `yield`.

Reads from the already-populated DB — fetchers already ran (README Quick Start).

## Chunking

`ingest/build_vector_store.py` chunks by data shape:

- **Wikipedia** (prose) → 4 sentences/chunk, 1-sentence overlap (`src/chunking.py`).
- **World Bank / FRED / SEC EDGAR** (facts) → no chunking; each row templated into one sentence, e.g. `"GDP for United States in 2025 was 30769700000000.00."`

The splitter is a plain regex (no NLTK) — sharp edge with citations:

In [8]:
from src.chunking import split_sentences

citation = (
    'Clifford Cobb, Ted Halstead and Jonathan Rowe. "If the GDP is up, '
    'why is America down?" The Atlantic Monthly, vol. 276, no. 4, '
    "October 1995, pages 59-78"
)
split_sentences(citation)


['Clifford Cobb, Ted Halstead and Jonathan Rowe. "If the GDP is up, why is America down?" The Atlantic Monthly, vol. 276, no. 4, October 1995, pages 59-78']

One sentence, not three. A naive `[.!?]\s+[A-Z0-9]` regex would also split after `vol.` and `no.`, tearing `vol. 276, no. 4` apart. `src/chunking.py` whitelists abbreviations (`vol`, `no`, `pp`, `e.g`, `i.e`, ...) and checks the preceding word first.

## Embedding

Every chunk — sentence or templated fact — goes through the same embedding call:

In [9]:
from src.embeddings import embed_texts

vectors = embed_texts(["Apple's revenue for fiscal year 2025 was $416B."])
print("dimensions:", len(vectors[0]))
print("first 5 values:", vectors[0][:5])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dimensions: 384
first 5 values: [0.050689924508333206, 0.050583891570568085, 0.10011453926563263, -0.011838517151772976, -0.052752889692783356]


`sentence-transformers/all-MiniLM-L6-v2`, 384 dimensions, runs locally — no API cost, good enough for MVP-scale retrieval. Same model for every source, so facts and prose share one searchable vector space.

## Storage: Postgres + pgvector

One database, raw staging schemas plus the shared vector store (`documents` + `document_chunks`) — no separate vector DB. `document_chunks.embedding` is a `VECTOR(384)` column with an HNSW index; a generated `tsvector` column with a GIN index gives full-text search. Both already in place, ready for Phase 3's hybrid search.

In [10]:
from src.db import get_conn

with get_conn() as conn, conn.cursor() as cur:
    cur.execute("SELECT source, count(*) FROM documents GROUP BY source ORDER BY source")
    print("documents by source:", cur.fetchall())
    cur.execute("SELECT count(*) FROM document_chunks")
    print("total chunks:", cur.fetchone()[0])


documents by source: [('fred', 5), ('secedgar', 24), ('wikipedia', 29), ('worldbank', 24)]
total chunks: 6846


In [11]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT content, metadata
        FROM document_chunks
        WHERE metadata->>'source' = 'secedgar'
        LIMIT 3
    """)
    for content, metadata in cur.fetchall():
        print(metadata)
        print(" ", content)
        print()


{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $59,531,000,000 for fiscal year ending 2018-09-29 (10-K filed 2020-10-30).

{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $55,256,000,000 for fiscal year ending 2019-09-28 (10-K filed 2021-10-29).

{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $57,411,000,000 for fiscal year ending 2020-09-26 (10-K filed 2022-10-28).



## Running the full pipeline yourself

Full ingestion, 5 commands (README Quick Start):

```bash
docker compose up -d postgres
uv run python -m ingest.fetch_wikipedia
uv run python -m ingest.fetch_worldbank
uv run python -m ingest.fetch_fred        # needs FRED_API_KEY
uv run python -m ingest.fetch_secedgar
uv run python -m ingest.build_vector_store
```

Fetchers are safe to re-run (dlt merges by key). `build_vector_store` truncates `documents`/`document_chunks` before rebuilding — without it, running it twice silently doubled every row (`documents.title` has no unique constraint). Fine for now; will need to become an upsert once Phase 2's Airflow DAG runs this daily.

## Recap

- 4 sources, each a small dlt resource → own raw Postgres schema.
- Wikipedia sentence-chunked; the other 3 templated one-fact-one-chunk.
- Everything embeds through one local model into one 384-dim space.
- One Postgres DB holds raw + vector store, HNSW and full-text indexes ready for later.
- Gotchas: dlt's silent type-variant columns, SEC EDGAR's XBRL tag drift, citation-splitting, a missing truncate that duplicated the store.

Phase 1 output: a populated, reproducible knowledge base. Phase 2 (claim extraction + RAG) builds on it.